In [4]:
#!/usr/bin/env python3
"""
First Link Analysis Function for MFSR Documents - CORRECTED VERSION
Identifies ONLY the first link that appears beneath each Project Title.
"""

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import sys
import traceback
import time

BASE = "https://www.mfsr.sk"

# All sector URLs to scrape
SECTOR_URLS = {
    "Obrana": "https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/obrana.html",
    "Budovy": "https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/budovy.html", 
    "Doprava": "https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/doprava.html",
    "Informatizacia": "https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/informatizacia.html",
    "Ostatne": "https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/ostatne.html"
}

# Regexes
date_re = re.compile(r'\b\d{1,2}\.\d{1,2}\.\d{4}\b')
size_re = re.compile(r'\b\d+(?:[.,]\d+)?\s*(?:kB|KB|MB|GB|B|kb|mb|gb)\b')

def extract_date_and_size(anchor):
    """Search for date and size strings around the anchor."""
    search_texts = []

    try:
        search_texts.append(anchor.get_text(" ", strip=True))
    except Exception:
        pass

    if anchor.parent:
        try:
            search_texts.append(anchor.parent.get_text(" ", strip=True))
        except Exception:
            pass

    for sib in list(anchor.previous_siblings)[:12]:
        if hasattr(sib, "get_text"):
            search_texts.append(sib.get_text(" ", strip=True))
        else:
            search_texts.append(str(sib).strip())

    for sib in list(anchor.next_siblings)[:6]:
        if hasattr(sib, "get_text"):
            search_texts.append(sib.get_text(" ", strip=True))
        else:
            search_texts.append(str(sib).strip())

    combined = " ".join([t for t in search_texts if t])
    date_match = date_re.search(combined)
    size_match = size_re.search(combined)

    return date_match.group(0) if date_match else None, size_match.group(0) if size_match else None

def is_document_link(href, link_text):
    """Check if link is a document (PDF, Word, Excel, PowerPoint, etc.)."""
    if not href:
        return False
    
    href_l = href.lower()
    link_text_l = link_text.lower()
    
    # Common document extensions
    document_extensions = [
        '.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx',
        '.txt', '.rtf', '.odt', '.ods', '.odp', '.csv', '.xml',
        '.zip', '.rar', '.7z', '.tar', '.gz'
    ]
    
    # Check for extensions in href
    for ext in document_extensions:
        if ext in href_l:
            return True
    
    # Check for document keywords in link text
    document_keywords = [
        'pdf', 'document', 'doc', 'excel', 'spreadsheet', 'presentation',
        'powerpoint', 'word', 'text', 'file', 'download', 'attachment'
    ]
    
    for keyword in document_keywords:
        if keyword in link_text_l:
            return True
    
    return False

def determine_document_type(link_text, parent_text=""):
    """Determine document type from link text and parent context."""
    text_to_search = f"{link_text} {parent_text}".lower()
    
    # Standard types
    if 'hodnotenie' in text_to_search:
        return 'hodnotenie'
    elif 'analýza' in text_to_search or 'analyza' in text_to_search:
        return 'analýza'
    elif 'štúdia uskutočniteľnosti' in text_to_search:
        return 'štúdia uskutočniteľnosti'
    elif 'aktualizácia' in text_to_search:
        return 'aktualizácia'
    elif 'stanovisko' in text_to_search:
        return 'stanovisko'
    elif 'správa' in text_to_search:
        return 'správa'
    elif 'metodika' in text_to_search:
        return 'metodika'
    elif 'zmluva' in text_to_search:
        return 'zmluva'
    elif 'dohoda' in text_to_search:
        return 'dohoda'
    elif 'investičný zámer' in text_to_search:
        return 'investičný zámer'
    else:
        # If no specific type found, use the link text itself (truncated)
        return link_text[:50] + "..." if len(link_text) > 50 else link_text

def get_first_link_per_project(sector_name, page_url, session):
    """Get ONLY the first link that appears beneath each project title."""
    print(f"\nAnalyzing {sector_name} sector: {page_url}")
    
    try:
        resp = session.get(page_url, timeout=20)
        if resp.status_code != 200:
            print(f"Non-200 status code for {sector_name}: {resp.status_code}")
            return []

        resp.encoding = resp.apparent_encoding or 'utf-8'
        soup = BeautifulSoup(resp.text, "lxml")

        project_links = []
        current_project = None
        found_first_link = False
        
        # Loop through all elements in order
        for elem in soup.find_all(["h4", "h5", "a"]):
            if elem.name == "h5":
                # This is a project title - reset for new project
                current_project = elem.get_text(" ", strip=True)
                found_first_link = False  # Reset flag for new project
                print(f"  Found project: {current_project}")
                
            elif elem.name == "a" and is_document_link(elem.get("href"), elem.get_text(" ", strip=True)):
                # This is a document link
                if current_project and not found_first_link:
                    # This is the FIRST link for this project
                    link_text = elem.get_text(" ", strip=True)
                    href = elem.get("href")
                    url = urljoin(BASE, href)
                    
                    # Determine document type
                    parent_text = elem.parent.get_text(" ", strip=True) if elem.parent else ""
                    dtype = determine_document_type(link_text, parent_text)
                    
                    # Extract date and size
                    date, size = extract_date_and_size(elem)
                    
                    project_links.append({
                        'Sector': sector_name,
                        'Project_Name': current_project,
                        'First_Link_Text': link_text,
                        'First_Link_URL': url,
                        'First_Link_Type': dtype,
                        'Date': date or "",
                        'File_Size': size or ""
                    })
                    
                    print(f"    ✓ First link: {link_text} ({dtype})")
                    found_first_link = True  # Mark that we found the first link
                    
                elif current_project and found_first_link:
                    # This is NOT the first link - skip it
                    print(f"    - Skipping additional link: {elem.get_text(' ', strip=True)}")
                    continue

        print(f"Found {len(project_links)} projects with first links in {sector_name}")
        return project_links

    except Exception as e:
        print(f"Error analyzing {sector_name}: {e}")
        return []

def analyze_first_links():
    """Main function to analyze first links for all projects."""
    try:
        session = requests.Session()
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/117.0.0.0 Safari/537.36",
            "Accept-Language": "en-US,en;q=0.9,sk;q=0.8"
        }
        session.headers.update(headers)

        all_first_links = []
        
        # Analyze each sector
        for sector_name, page_url in SECTOR_URLS.items():
            sector_links = get_first_link_per_project(sector_name, page_url, session)
            all_first_links.extend(sector_links)
            
            # Small delay between requests
            time.sleep(1)

        # Create DataFrame
        df_first_links = pd.DataFrame(all_first_links)
        
        print(f"\n{'='*60}")
        print(f"FIRST LINK ANALYSIS COMPLETE")
        print(f"{'='*60}")
        print(f"Total projects analyzed: {len(df_first_links)}")
        
        # Show summary by sector
        print(f"\nProjects by sector:")
        sector_counts = df_first_links['Sector'].value_counts()
        for sector, count in sector_counts.items():
            print(f"  {sector}: {count}")
        
        # Show document types of first links
        print(f"\nFirst link document types:")
        type_counts = df_first_links['First_Link_Type'].value_counts()
        for doc_type, count in type_counts.items():
            print(f"  {doc_type}: {count}")
        
        # Show sample results
        print(f"\nSample of first links:")
        print(df_first_links[['Sector', 'Project_Name', 'First_Link_Type', 'First_Link_Text']].head(10).to_string(index=False))
        
        # Save to CSV
        output_file = "first_link.csv"
        df_first_links.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"\nSaved first link analysis to: {output_file}")
        
        return df_first_links

    except Exception as e:
        print("Exception occurred:", e)
        traceback.print_exc()
        return None

# Run the analysis
if __name__ == "__main__":
    first_links_df = analyze_first_links()


Analyzing Obrana sector: https://www.mfsr.sk/sk/financie/hodnota-za-peniaze/hodnotenia/obrana.html
  Found project: Rekonštrukcia leteckej základne Sliač (aktualizácia pred podpisom zmluvy)
    ✓ First link: Hodnotenie (hodnotenie)
    - Skipping additional link: Aktualizovaná štúdia uskutočniteľnosti
  Found project: Rekonštrukcia leteckej základne Sliač (aktualizácia)
    ✓ First link: Hodnotenie (hodnotenie)
    - Skipping additional link: Aktualizovaná štúdia uskutočniteľnosti
  Found project: Obstaranie útočnej pušky 5,56 mm s príslušenstvom
    ✓ First link: Hodnotenie (hodnotenie)
    - Skipping additional link: Štúdia uskutočniteľnosti
  Found project: Akvizícia palebných prostriedkov protivzdušnej obrany – prvá etapa (aktualizácia)
    ✓ First link: Analýza (analýza)
    - Skipping additional link: Štúdia uskutočniteľnosti
  Found project: Obstarávanie stavby na leteckej základni Sliač
    ✓ First link: Analýza (analýza)
    - Skipping additional link: Štúdia uskutočniteľnost

In [9]:
#!/usr/bin/env python3
"""
First Link PDF Downloader
Matches first links from first_link.csv with full_mfsr_data_completewith_ids.csv
and downloads only those specific PDFs using Document_ID as filename.
"""

import pandas as pd
import requests
import os
import time
from urllib.parse import urlparse
import traceback
from pathlib import Path

def setup_download_directory():
    """Create downloads directory if it doesn't exist."""
    download_dir = Path("downloaded_first_links")
    download_dir.mkdir(exist_ok=True)
    return download_dir

def clean_filename(filename):
    """Clean filename to be filesystem-safe."""
    # Remove or replace invalid characters
    invalid_chars = '<>:"/\\|?*'
    for char in invalid_chars:
        filename = filename.replace(char, '_')
    
    # Remove extra spaces and dots
    filename = filename.strip('. ')
    
    # Ensure it's not too long
    if len(filename) > 200:
        filename = filename[:200]
    
    return filename

def get_file_extension(url):
    """Extract file extension from URL."""
    parsed_url = urlparse(url)
    path = parsed_url.path.lower()
    
    # Common document extensions
    extensions = ['.pdf', '.doc', '.docx', '.xls', '.xlsx', '.ppt', '.pptx',
                 '.txt', '.rtf', '.odt', '.ods', '.odp', '.csv', '.xml',
                 '.zip', '.rar', '.7z', '.tar', '.gz']
    
    for ext in extensions:
        if path.endswith(ext):
            return ext
    
    # Default to .pdf if no extension found
    return '.pdf'

def is_archive_file(url):
    """Check if the URL points to an archive file that should be skipped."""
    parsed_url = urlparse(url)
    path = parsed_url.path.lower()
    
    # Archive extensions to skip
    archive_extensions = ['.zip', '.rar', '.7z', '.tar', '.gz']
    
    for ext in archive_extensions:
        if path.endswith(ext):
            return True
    
    return False

def download_document(url, filepath, session, timeout=30):
    """Download a document from URL and save to filepath."""
    try:
        print(f"Downloading: {url}")
        
        response = session.get(url, timeout=timeout, stream=True)
        response.raise_for_status()
        
        # Check content type for validation
        content_type = response.headers.get('content-type', '').lower()
        print(f"  Content-Type: {content_type}")
        
        # Save the file
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        
        # Verify file was created and has content
        if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
            file_size = os.path.getsize(filepath)
            print(f"  ✓ Success: {file_size:,} bytes")
            return True, f"Success: {file_size:,} bytes"
        else:
            print(f"  ✗ Failed: File is empty or doesn't exist")
            return False, "Failed: File is empty"
            
    except requests.exceptions.Timeout:
        error_msg = "Timeout"
        print(f"  ✗ Failed: {error_msg}")
        return False, error_msg
    except requests.exceptions.ConnectionError:
        error_msg = "Connection error"
        print(f"  ✗ Failed: {error_msg}")
        return False, error_msg
    except requests.exceptions.HTTPError as e:
        error_msg = f"HTTP error: {e.response.status_code}"
        print(f"  ✗ Failed: {error_msg}")
        return False, error_msg
    except Exception as e:
        error_msg = f"Unexpected error: {str(e)}"
        print(f"  ✗ Failed: {error_msg}")
        return False, error_msg

def match_first_links_with_documents():
    """Match first links with full dataset and extract Document IDs."""
    print("="*60)
    print("MATCHING FIRST LINKS WITH DOCUMENT IDs")
    print("="*60)
    
    # Load first links data
    try:
        first_links_df = pd.read_csv('first_link.csv')
        print(f"Loaded {len(first_links_df)} first links from first_link.csv")
    except FileNotFoundError:
        print("Error: first_link.csv not found!")
        return None
    
    # Load full dataset
    try:
        full_df = pd.read_csv('full_mfsr_data_completewith_ids.csv')
        print(f"Loaded {len(full_df)} documents from full_mfsr_data_completewith_ids.csv")
    except FileNotFoundError:
        print("Error: full_mfsr_data_completewith_ids.csv not found!")
        return None
    
    # Match first links with full dataset
    matched_documents = []
    
    print(f"\nMatching first links with documents...")
    print("-" * 60)
    
    for idx, first_link in first_links_df.iterrows():
        project_name = first_link['Project_Name']
        first_link_url = first_link['First_Link_URL']
        sector = first_link['Sector']
        
        # Find matching documents in full dataset
        # Match by Project Name and URL
        matches = full_df[
            (full_df['Project Name'] == project_name) & 
            (full_df['URL'] == first_link_url)
        ]
        
        if len(matches) > 0:
            # Take the first match (should be unique)
            match = matches.iloc[0]
            matched_documents.append({
                'Document_ID': match['Document_ID'],
                'Sector': sector,
                'Project_Name': project_name,
                'Type': match['Type'],
                'URL': first_link_url,
                'First_Link_Text': first_link['First_Link_Text'],
                'First_Link_Type': first_link['First_Link_Type'],
                'Date': match['Date'],
                'File_Size': match['File Size']
            })
            print(f"✓ Matched: {project_name} -> {match['Document_ID']}")
        else:
            print(f"✗ No match found for: {project_name}")
            # Try alternative matching by URL only
            url_matches = full_df[full_df['URL'] == first_link_url]
            if len(url_matches) > 0:
                match = url_matches.iloc[0]
                matched_documents.append({
                    'Document_ID': match['Document_ID'],
                    'Sector': sector,
                    'Project_Name': match['Project Name'],  # Use the project name from full dataset
                    'Type': match['Type'],
                    'URL': first_link_url,
                    'First_Link_Text': first_link['First_Link_Text'],
                    'First_Link_Type': first_link['First_Link_Type'],
                    'Date': match['Date'],
                    'File_Size': match['File Size']
                })
                print(f"✓ Matched by URL: {match['Project Name']} -> {match['Document_ID']}")
            else:
                print(f"✗ No URL match found for: {first_link_url}")
    
    matched_df = pd.DataFrame(matched_documents)
    print(f"\nSuccessfully matched {len(matched_df)} documents")
    
    return matched_df

def download_first_links():
    """Main function to download first link PDFs."""
    
    # Match first links with documents
    matched_df = match_first_links_with_documents()
    if matched_df is None or len(matched_df) == 0:
        print("No documents to download!")
        return
    
    # Filter out documents without URLs
    df_with_urls = matched_df[matched_df['URL'].notna() & (matched_df['URL'] != '')]
    print(f"Documents with URLs: {len(df_with_urls)}")
    
    if len(df_with_urls) == 0:
        print("No documents with URLs found!")
        return
    
    # Setup download directory
    download_dir = setup_download_directory()
    print(f"Download directory: {download_dir.absolute()}")
    
    # Setup session
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9,sk;q=0.8"
    })
    
    # Track download results
    download_results = []
    
    print(f"\nStarting downloads...")
    print("-" * 60)
    
    # Download each document
    for idx, row in df_with_urls.iterrows():
        doc_id = row['Document_ID']
        url = row['URL']
        sector = row['Sector']
        project_name = row['Project_Name']
        doc_type = row['Type']
        first_link_text = row['First_Link_Text']
        
        # Check if this is an archive file that should be skipped
        if is_archive_file(url):
            print(f"Skipping {doc_id} - archive file (.zip, .rar, etc.)")
            download_results.append({
                'Document_ID': doc_id,
                'Sector': sector,
                'Project_Name': project_name,
                'Type': doc_type,
                'First_Link_Text': first_link_text,
                'URL': url,
                'Filename': f"{doc_id}{get_file_extension(url)}",
                'Status': 'Failed',
                'Message': 'Skipped: Archive file (.zip, .rar, etc.)',
                'File_Size': 0
            })
            continue
        
        # Create filename from Document_ID with proper extension
        file_extension = get_file_extension(url)
        filename = f"{doc_id}{file_extension}"
        filename = clean_filename(filename)
        filepath = download_dir / filename
        
        # Skip if file already exists
        if filepath.exists():
            print(f"Skipping {doc_id} - file already exists")
            download_results.append({
                'Document_ID': doc_id,
                'Sector': sector,
                'Project_Name': project_name,
                'Type': doc_type,
                'First_Link_Text': first_link_text,
                'URL': url,
                'Filename': filename,
                'Status': 'Skipped',
                'Message': 'File already exists',
                'File_Size': os.path.getsize(filepath)
            })
            continue
        
        # Download the document
        success, message = download_document(url, filepath, session)
        
        # Record result
        file_size = os.path.getsize(filepath) if success and filepath.exists() else 0
        download_results.append({
            'Document_ID': doc_id,
            'Sector': sector,
            'Project_Name': project_name,
            'Type': doc_type,
            'First_Link_Text': first_link_text,
            'URL': url,
            'Filename': filename,
            'Status': 'Success' if success else 'Failed',
            'Message': message,
            'File_Size': file_size
        })
        
        # Small delay to be respectful to the server
        time.sleep(0.5)
    
    # Create results DataFrame
    results_df = pd.DataFrame(download_results)
    
    # Save results CSV
    results_file = "first_link_download_results.csv"
    results_df.to_csv(results_file, index=False, encoding="utf-8-sig")
    print(f"\nDownload results saved to: {results_file}")
    
    # Save separate CSV for failed downloads only
    failed_df = results_df[results_df['Status'] == 'Failed']
    if len(failed_df) > 0:
        failed_file = "first_link_failed_downloads.csv"
        failed_df.to_csv(failed_file, index=False, encoding="utf-8-sig")
        print(f"Failed downloads saved to: {failed_file}")
    
    # Print summary
    print("\n" + "="*60)
    print("FIRST LINK DOWNLOAD SUMMARY")
    print("="*60)
    
    total_docs = len(results_df)
    successful = len(results_df[results_df['Status'] == 'Success'])
    failed = len(results_df[results_df['Status'] == 'Failed'])
    skipped = len(results_df[results_df['Status'] == 'Skipped'])
    
    print(f"Total first link documents: {total_docs}")
    print(f"Successfully downloaded: {successful}")
    print(f"Failed downloads: {failed}")
    print(f"Skipped (already exists): {skipped}")
    print(f"Success rate: {(successful/total_docs)*100:.1f}%")
    
    # Show failed downloads with more detail
    if failed > 0:
        print(f"\nFailed downloads details:")
        failed_docs = results_df[results_df['Status'] == 'Failed']
        for _, row in failed_docs.iterrows():
            print(f"  {row['Document_ID']} ({row['Sector']}): {row['Message']}")
            print(f"    URL: {row['URL']}")
    
    # Show status breakdown by sector
    print(f"\nStatus breakdown by sector:")
    sector_status = results_df.groupby(['Sector', 'Status']).size().unstack(fill_value=0)
    print(sector_status)
    
    # Show file size statistics
    successful_docs = results_df[results_df['Status'].isin(['Success', 'Skipped'])]
    if len(successful_docs) > 0:
        total_size = successful_docs['File_Size'].sum()
        avg_size = successful_docs['File_Size'].mean()
        print(f"\nFile size statistics:")
        print(f"Total downloaded size: {total_size:,} bytes ({total_size/1024/1024:.1f} MB)")
        print(f"Average file size: {avg_size:,.0f} bytes ({avg_size/1024:.1f} KB)")
    
    print(f"\nDownloaded files are in: {download_dir.absolute()}")

if __name__ == "__main__":
    download_first_links()

MATCHING FIRST LINKS WITH DOCUMENT IDs
Loaded 267 first links from first_link.csv
Loaded 354 documents from full_mfsr_data_completewith_ids.csv

Matching first links with documents...
------------------------------------------------------------
✓ Matched: Rekonštrukcia leteckej základne Sliač (aktualizácia pred podpisom zmluvy) -> 4.016
✓ Matched: Rekonštrukcia leteckej základne Sliač (aktualizácia) -> 4.015
✓ Matched: Obstaranie útočnej pušky 5,56 mm s príslušenstvom -> 4.014
✓ Matched: Akvizícia palebných prostriedkov protivzdušnej obrany – prvá etapa (aktualizácia) -> 4.013
✓ Matched: Obstarávanie stavby na leteckej základni Sliač -> 4.012
✓ Matched: Akvizícia prostriedkov protivzdušnej obrany (projektový zámer) -> 4.011
✓ Matched: Obstaranie Pásových bojových obrnených vozidiel a Pásových obrnených vozidiel (aktualizácia) -> 4.01
✓ Matched: Aktualizácia hodnotenia projektu Bojových obrnených vozidiel 8x8 -> 4.009
✓ Matched: Obstaranie Pásových bojových obrnených vozidiel a Pásových

In [12]:
#!/usr/bin/env python3
"""
Create tagged dataset and project summary
1. Tags documents as First File/MoF Assessment (1) or not (0)
2. Creates project summary with document counts, types, and Document IDs
"""

import pandas as pd
import numpy as np

def create_tagged_dataset():
    """Create tagged dataset from full_mfsr_data_completewith_ids.csv"""
    print("="*60)
    print("CREATING TAGGED DATASET")
    print("="*60)
    
    # Load datasets
    try:
        full_df = pd.read_csv('full_mfsr_data_completewith_ids.csv')
        print(f"Loaded {len(full_df)} documents from full_mfsr_data_completewith_ids.csv")
    except FileNotFoundError:
        print("Error: full_mfsr_data_completewith_ids.csv not found!")
        return None
    
    try:
        first_links_df = pd.read_csv('first_link.csv')
        print(f"Loaded {len(first_links_df)} first links from first_link.csv")
    except FileNotFoundError:
        print("Error: first_link.csv not found!")
        return None
    
    # Get list of first link URLs
    first_link_urls = set(first_links_df['First_Link_URL'].tolist())
    print(f"Found {len(first_link_urls)} unique first link URLs")
    
    # Create tagged dataset
    tagged_df = full_df.copy()
    
    # Tag documents: 1 if it's a first link, 0 otherwise
    tagged_df['First_File_MoF_Assessment'] = tagged_df['URL'].apply(
        lambda url: 1 if url in first_link_urls else 0
    )
    
    # Count tagged documents
    first_files_count = tagged_df['First_File_MoF_Assessment'].sum()
    other_files_count = len(tagged_df) - first_files_count
    
    print(f"\nTagging results:")
    print(f"  First File/MoF Assessment (1): {first_files_count}")
    print(f"  Other documents (0): {other_files_count}")
    
    # Save tagged dataset
    output_file = "tagged_mfsr_data.csv"
    tagged_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\nSaved tagged dataset to: {output_file}")
    
    return tagged_df

def create_project_summary():
    """Create project summary with document counts, types, and Document IDs"""
    print("\n" + "="*60)
    print("CREATING PROJECT SUMMARY")
    print("="*60)
    
    # Load tagged dataset
    try:
        tagged_df = pd.read_csv('tagged_mfsr_data.csv')
        print(f"Loaded {len(tagged_df)} documents from tagged_mfsr_data.csv")
    except FileNotFoundError:
        print("Error: tagged_mfsr_data.csv not found!")
        return None
    
    # Load first links for reference
    try:
        first_links_df = pd.read_csv('first_link.csv')
        print(f"Loaded {len(first_links_df)} first links from first_link.csv")
    except FileNotFoundError:
        print("Error: first_link.csv not found!")
        return None
    
    # Create project summary
    project_summaries = []
    
    # Group by project name
    for project_name, project_group in tagged_df.groupby('Project Name'):
        # Get first file information
        first_file = project_group[project_group['First_File_MoF_Assessment'] == 1]
        
        if len(first_file) > 0:
            # There is a first file for this project
            first_file_info = first_file.iloc[0]  # Take the first one (should be unique)
            
            # Get other documents (not first file)
            other_docs = project_group[project_group['First_File_MoF_Assessment'] == 0]
            
            # Count documents
            total_docs = len(project_group)
            other_docs_count = len(other_docs)
            
            # Get types of other documents (excluding first file)
            if len(other_docs) > 0:
                other_types = other_docs['Type'].unique().tolist()
                # Filter out NaN values and convert to strings
                other_types = [str(t) for t in other_types if pd.notna(t)]
                other_types_str = ', '.join(sorted(other_types))
            else:
                other_types_str = "None"
            
            # Get Document IDs of other documents (excluding first file)
            if len(other_docs) > 0:
                other_doc_ids = other_docs['Document_ID'].tolist()
                # Filter out NaN values and convert to strings
                other_doc_ids = [str(doc_id) for doc_id in other_doc_ids if pd.notna(doc_id)]
                other_doc_ids_str = ', '.join(sorted(other_doc_ids))
            else:
                other_doc_ids_str = "None"
            
            project_summaries.append({
                'Project_Name': project_name,
                'First_File_Document_ID': first_file_info['Document_ID'],
                'First_File_URL': first_file_info['URL'],
                'First_File_Type': first_file_info['Type'],
                'Total_Documents_in_Project': total_docs,
                'Other_Documents_Count': other_docs_count,
                'Other_Document_Types': other_types_str,
                'Other_Document_IDs': other_doc_ids_str,
                'Sector': first_file_info['Sector']
            })
        else:
            # No first file found for this project (shouldn't happen based on our logic)
            print(f"Warning: No first file found for project: {project_name}")
    
    # Create DataFrame
    project_summary_df = pd.DataFrame(project_summaries)
    
    print(f"\nCreated summary for {len(project_summary_df)} projects")
    
    # Show sample results
    print(f"\nSample project summaries:")
    sample_cols = ['Project_Name', 'First_File_Document_ID', 'Total_Documents_in_Project', 'Other_Document_Types', 'Other_Document_IDs']
    print(project_summary_df[sample_cols].head(10).to_string(index=False))
    
    # Show statistics
    print(f"\nProject statistics:")
    print(f"  Total projects: {len(project_summary_df)}")
    print(f"  Average documents per project: {project_summary_df['Total_Documents_in_Project'].mean():.1f}")
    print(f"  Projects with only 1 document: {len(project_summary_df[project_summary_df['Total_Documents_in_Project'] == 1])}")
    print(f"  Projects with multiple documents: {len(project_summary_df[project_summary_df['Total_Documents_in_Project'] > 1])}")
    
    # Show document type distribution for first files
    print(f"\nFirst file document types:")
    first_file_types = project_summary_df['First_File_Type'].value_counts()
    for doc_type, count in first_file_types.items():
        print(f"  {doc_type}: {count}")
    
    # Show examples of projects with multiple documents
    multi_doc_projects = project_summary_df[project_summary_df['Total_Documents_in_Project'] > 1]
    if len(multi_doc_projects) > 0:
        print(f"\nExamples of projects with multiple documents:")
        example_cols = ['Project_Name', 'First_File_Document_ID', 'Other_Document_IDs', 'Other_Document_Types']
        print(multi_doc_projects[example_cols].head(5).to_string(index=False))
    
    # Save project summary
    output_file = "project_summary.csv"
    project_summary_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\nSaved project summary to: {output_file}")
    
    return project_summary_df

def main():
    """Main function to create both datasets"""
    print("MFSR DATA PROCESSING")
    print("="*60)
    
    # Step 1: Create tagged dataset
    tagged_df = create_tagged_dataset()
    if tagged_df is None:
        return
    
    # Step 2: Create project summary
    project_summary_df = create_project_summary()
    if project_summary_df is None:
        return
    
    print("\n" + "="*60)
    print("PROCESSING COMPLETE")
    print("="*60)
    print("Files created:")
    print("  1. tagged_mfsr_data.csv - Full dataset with First File/MoF Assessment tags")
    print("  2. project_summary.csv - Project-level summary with document counts, types, and Document IDs")

if __name__ == "__main__":
    main()

MFSR DATA PROCESSING
CREATING TAGGED DATASET
Loaded 354 documents from full_mfsr_data_completewith_ids.csv
Loaded 267 first links from first_link.csv
Found 267 unique first link URLs

Tagging results:
  First File/MoF Assessment (1): 267
  Other documents (0): 87

Saved tagged dataset to: tagged_mfsr_data.csv

CREATING PROJECT SUMMARY
Loaded 354 documents from tagged_mfsr_data.csv
Loaded 267 first links from first_link.csv

Created summary for 265 projects

Sample project summaries:
                                                                                                                Project_Name  First_File_Document_ID  Total_Documents_in_Project     Other_Document_Types Other_Document_IDs
                                                             Administratívna budova Úradu vlády Slovenskej republiky „Čajka“                   3.006                           1                     None               None
                                                            Aktualizác